# SFT + GRPO v2 on GSM8K (Qwen3-0.6B)
Click **Save Version → Save & Run All** to run as a background job. Safe to close your computer.

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────
import shutil, os, sys, subprocess

os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/Fine-tuning-and-GRPO-on-Qwen', ignore_errors=True)
subprocess.run(['git', 'clone',
    'https://github.com/202520030411/Fine-tuning-and-GRPO-on-Qwen.git',
    '/kaggle/working/Fine-tuning-and-GRPO-on-Qwen'], check=True)

REPO = '/kaggle/working/Fine-tuning-and-GRPO-on-Qwen'
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ['HF_HOME']            = '/kaggle/working/.hf'
os.environ['HF_DATASETS_CACHE']  = '/kaggle/working/.hf/datasets'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/.hf/transformers'
print('Repo ready')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers>=4.51.0', 'peft>=0.11.1', 'datasets>=2.19.0',
    'accelerate', 'tqdm', 'typer', 'sentencepiece', 'safetensors', 'matplotlib'
], check=True)
print('Done')

In [ ]:
# ── 3. Download model + prepare dataset ──────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B-Base')
AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B-Base')
print('Model cached')

In [ ]:
!python scripts/prepare_gsm8k.py

In [ ]:
# ── 4. SFT — skip if already done ────────────────────────────────────────
import os
if not os.path.exists('/kaggle/working/model/sft_gsm8k/adapter_model.safetensors'):
    print('Training SFT...')
    os.system("""python scripts/train_sft.py \
        --model-name-or-path Qwen/Qwen3-0.6B-Base \
        --train-path dataset/processed/gsm8k_train.jsonl \
        --output-dir /kaggle/working/model/sft_gsm8k \
        --train-log-path /kaggle/working/model/sft_gsm8k/train_log.jsonl \
        --max-steps 300 --per-device-batch-size 2 --grad-accum 16 \
        --max-length 384 --fp16 --gradient-checkpointing --log-every 10""")
else:
    print('SFT already done, skipping')

In [ ]:
# ── 5. Eval base — skip if already done ──────────────────────────────────
import os
if not os.path.exists('/kaggle/working/model/eval_base.jsonl'):
    print('Evaluating base model...')
    os.system("""python scripts/eval.py \
        --base-model Qwen/Qwen3-0.6B-Base \
        --test-path dataset/processed/gsm8k_test.jsonl \
        --output-path /kaggle/working/model/eval_base.jsonl \
        --max-new-tokens 256 --batch-size 8""")
else:
    print('Base eval already done, skipping')

In [ ]:
# ── 6. Eval SFT — skip if already done ───────────────────────────────────
import os
if not os.path.exists('/kaggle/working/model/eval_sft.jsonl'):
    print('Evaluating SFT model...')
    os.system("""python scripts/eval.py \
        --base-model Qwen/Qwen3-0.6B-Base \
        --adapter-path /kaggle/working/model/sft_gsm8k \
        --test-path dataset/processed/gsm8k_test.jsonl \
        --output-path /kaggle/working/model/eval_sft.jsonl \
        --max-new-tokens 256 --batch-size 8""")
else:
    print('SFT eval already done, skipping')

In [ ]:
# ── 7. GRPO v2 (500 steps, group=8, kl=0.01) ─────────────────────────────
!python scripts/train_grpo.py \
    --model-name-or-path Qwen/Qwen3-0.6B-Base \
    --ref-model-path /kaggle/working/model/sft_gsm8k \
    --train-path dataset/processed/gsm8k_train.jsonl \
    --output-dir /kaggle/working/model/grpo_gsm8k_v2 \
    --train-log-path /kaggle/working/model/grpo_gsm8k_v2/train_log.jsonl \
    --max-steps 500 --group-size 8 --kl-coef 0.01 --log-every 5

In [ ]:
# ── 8. Eval GRPO v2 ───────────────────────────────────────────────────────
!python scripts/eval.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --adapter-path /kaggle/working/model/grpo_gsm8k_v2 \
    --test-path dataset/processed/gsm8k_test.jsonl \
    --output-path /kaggle/working/model/eval_grpo_v2.jsonl \
    --max-new-tokens 256 --batch-size 8

In [ ]:
# ── 9. Analyze + plots ────────────────────────────────────────────────────
!python scripts/analyze.py \
    --base-results  /kaggle/working/model/eval_base.jsonl \
    --sft-results   /kaggle/working/model/eval_sft.jsonl \
    --grpo-results  /kaggle/working/model/eval_grpo_v2.jsonl \
    --sft-log       /kaggle/working/model/sft_gsm8k/train_log.jsonl \
    --grpo-log      /kaggle/working/model/grpo_gsm8k_v2/train_log.jsonl \
    --images-dir    /kaggle/working/images

In [ ]:
# ── 10. Zip everything for download ──────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/model')
shutil.make_archive('/kaggle/working/plots',   'zip', '/kaggle/working/images')
print('ALL DONE — download results.zip and plots.zip from the Output panel')